In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# parameters
batch_size = 32       # 32 sequences at once
block_size = 8        # length of each seq = 8
steps = 3000          # 3000 training updates
learning_rate = 1e-2  # size of parameter updates

# importing dataset
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()     # saves as one python string

# creating vocabulary
chars = sorted(list(set(text)))     # gets unique characters and sorts them
vocab_size = len(chars)             # vocab_size = total no. of unique characters

# dictionaries
stoi = {ch: i for i, ch in enumerate(chars)} # enumerate assigns a number
itos = {i: ch for i, ch in enumerate(chars)}

# encoder - decoder
def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# encode entire dataset
data = torch.tensor(encode(text), dtype=torch.long)

# train - test split
n = int(0.9 * len(data))

train_data = data[:n]     # 90%
test_data = data[n:]      # 10%

# batching : goal - creating training examples

def get_batch(split):

    data = train_data if split == 'train' else test_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])

    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

# model

class BigramLanguageModel(nn.Module):

    # some syntax i didnt know about so had to gpt
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )


    # forward pass
    def forward(self, idx, targets=None):

        # idx: (B,T)
        logits = self.token_embedding_table(idx)
        # logits: (B,T,C) as every token predicts next token probabilities

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C) # flatteing as entropy wants (N,C) and (N)
            targets = targets.view(B * T)  # here N = B*T

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # gets predictions
            logits, loss = self(idx)

            # focus on last timestep
            logits = logits[:, -1, :]

            # logits convert to probabilities
            probs = F.softmax(logits, dim=-1)

            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)

            # append sampled token
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# initialize model
model = BigramLanguageModel(vocab_size)

# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# training loop
for iter in range(steps):

    # sampling batch
    xb, yb = get_batch('train')

    # forward pass
    logits, loss = model(xb, yb)

    # clear gradients
    optimizer.zero_grad(set_to_none=True)

    # backprop
    loss.backward()

    # update parameters
    optimizer.step()

    # print loss
    if iter % 300 == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

# generate text
context = torch.zeros((1, 1), dtype=torch.long)

generated_tokens = model.generate(
    context,
    max_new_tokens=500
)

generated_text = decode(
    generated_tokens[0].tolist()
)

print("\nGenerated Text:\n")
print(generated_text)

step 0: loss 4.5700
step 300: loss 2.7259
step 600: loss 2.5765
step 900: loss 2.5231
step 1200: loss 2.4169
step 1500: loss 2.4622
step 1800: loss 2.4788
step 2100: loss 2.4714
step 2400: loss 2.5815
step 2700: loss 2.4506

Generated Text:


III f ARMNoo ke w anyoro ul p theINNURIO fred th 's ot thorswalalld nd atoup'sle yothe'd, me more,
Wh iose,
DUBur f oulerand! Y:
O ist s oomeren wnethare anthy d ben deane mertroud VOREd t s,

Th ghind hiogovet le im'etist, ant
RCAPOLom maloman aur pl sthino t.
be trareand wise se.

CKI I, thoumiruen wnnoffote, omer neace can
F tre thim
A:
O:
An nooremerowitho d-
I'dinoorveetthaisoribr voout, s
tstomst dinos
arththsheneaiget hitonordereresad g erond ad, hak tcalysp llystathe re abl oceasIOLYored


# Some Notes from stuff I understood

every row of table holds scores for next token.

for example - if token id = 10

then logits = embedding_table[10] returns "vocab_size" numbers


**FORWARD:**

suppose idx.shape = (4,8), i.e, (batch_size=4, block_size=8)

then output.shape = (4,8,vocab_size) = (B, T, C) = (batch, time/position, vocab_size)

For every token, model predicts distribution over vocab.


**RESHAPE:**

cross entropy expects (N,C) not (B,T,C)

so we flatten it by N = B*T

Each token prediction becomes one training example.

Targets also flatten: (B,T) -> (N)

**CROSS ENTROPY:**

internally applies softmax

compares with correct answers

computes average negative log likelihood
